# Welcome to DOWNFLOWGO 🌋
---

A number of numerical models exist to support lava flow modeling (see review by Dietterich et al. [2017]). Here, we combine the stochastic model DOWNFLOW [Favalli et al. 2005](https://doi.org/10.1029/2004gl021718) with the deterministic model FLOWGO [Harris and Rowland 2001](https://doi.org/10.1007/s004450000120) to support hazard assessment at Piton de la Fournaise , mainly.

DOWNFLOW provides the most likely lava flow paths, including the LoSD and area of coverage. For responding to current crises at Piton de la Fournaise, this model was calibrated by fitting the output flow coverage to the actual areas of all flow fields since 2016 [Chevrel et al. 2021](https://doi.org/10.5194/nhess-21-2355-2021). DOWNFLOW must be calibrated (dh and N parameters) for you volcano.

FLOWGO calculates the runout distance of a lava flow along a slope line for a given effusion rate [Harris and Rowland 2001](https://doi.org/10.1007/s004450000120). It is 1-D model adapted for cooling-limited basaltic lava flow in a channel of uniform depth that, once calibrated with a suitable models and input parameters, only needs the slope from the vent along the steepest descent line and a discharge rate as its source terms. The code is now an open access python library : [PyFLOwGO](https://github.com/pyflowgo/pyflowgo.git) presented in [Chevrel et al. 2018](https://doi.org/10.1016/j.cageo.2017.11.009).

FLOWGO source terms must be adapted accordingly to your lava flow. See exemples on Earth: Hawaiʻi, (Harris et al., 2022; Harris & Rowland, 2001, 2015; Mossoux et al., 2016; Riker et al., 2009; Robert et al., 2014; Rowland et al., 2005; Chevrel et al., 2018; Thompson and Ramsey, 2021), Italy (Harris et al., 2007, 2011; Wright et al., 2008) ; Kamchatka (Ramsey et al. 2019); Mt Cameroon (Wantim et al. 2013); D.R. Congo (Mossoux et al. 2016), Galapagos (Rowland et al. 2003) La Reunion (Chevrel et al., 2018, 2022; Harris et al., 2016, 2019; Peltier et al., 2022; Rhéty et al., 2017) and on other planets (see also next session): Mars (Flynn et al., 2022; Rowland et al., 2004) Mercury (Vetere et al. 2017) ; Moon (Lev et al. 2021) and Venus Flynn et al. (2023).

To use this package you must cite these 3 publications: [Chevrel et al. 2022](https://doi.org/10.30909/vol.05.02.313334), [Favalli 2005](https://doi.org/10.30909/vol.05.02.313334) and [Harris and Rowland 2001](https://doi.org/10.1007/s004450000120):

Chevrel MO, Harris A, Peltier A, Villeneuve N, Coppola D, Gouhier M, Drenne S. (2022) Volcanic crisis management supported by near real time lava flow hazard assessment at Piton de la Fournaise, Volcanica 5(2), pp. 313–334. https://doi.org/10.30909/vol.05.02.313334

Harris, A. J. L. and S. Rowland (2001). “FLOWGO: a kinematic thermo-rheological model for lava flowing in a channel”. Bulletin of Volcanology 63(1), pages 20–44. issn: 1432-0819. https://doi.org/10.1007/s004450000120.

Favalli, M. et al. (2005). “Forecasting lava flow paths by a stochastic approach”. Geophysical Research Letters 32(3). issn: 0094- 8276. https://doi.org/10.1029/2004gl021718.**

In [ ]:
import time
import downflowgo.datamanager as datamanager
from downflowgo.all_for_grid import grid_maker_reader
from downflowgo.config_loader import Config
from downflowgo.perf_timer import runtime
from downflowgo.runner import Runner
from downflowgo.mapping import Mapping
from pandas import DataFrame # Pour affcher proprement les tableaux dans les prints
from IPython.display import clear_output # Nettoie les cellules de Jupyter

In [ ]:
# Start the timer
start_time = time.time()

In [ ]:
# Initializes the configuration
config = Config()

In [ ]:
# Choose the language of the map
config.language = "FR"

## General configuration
`mode` can be:
 - **downflowgo**
 - **downflow**

`grid_mode` can be:
 - **yes**
    > In [**`all_for_grid.py`**](downflowgo/all_for_grid.py): **DOWNFLOWGO** defines an area around the pixel containing the vent location and discretizes it according to the user-specified resolution. A **DOWFLOW** model is then run for each vent defined in the grid. The model subsequently constructs a map showing the most probable lava flow paths.
 - **no**

`delete_existing` on **yes** will delete all results in the folder `path_to_folder`.

In [ ]:

# [config_general]
config.mode = "downflowgo"
config.grid_mode = "no"
config.delete_existing = "yes" # Delete existing results?

In [ ]:
# [paths]
config.path_to_eruptions = "./data/outputs" # Eruption folder
config.dem = "./data/inputs/DEM/MNT-post-20220919_5m.asc"
config.csv_vent_file = "0"

# If grid mode is activate else None
config.grid_csv = "./data/Vent_grid.csv"

# [pyflowgo]
config.json_input = "./data/inputs/PdF_template.json"

config.validate_path()

In [ ]:
file_opener = datamanager.DataManager(config)

In [ ]:
# [downflow]
config.name_vent = "pixel_center_2"
config.easting = 369082.7
config.northing = 7647204.29
config.DH = 2
config.n = 10000
config.slope_step = 10
config.epsg_code = 32740

# [pyflowgo]
config.effusion_rates_input = "5,30,5"
effusion_rates = config.set_effusion_rate(config.effusion_rates_input)
config.show(effusion_rates)

In [ ]:
# Writes a csv file with the position of the vent in the config file
file_opener.csv_vent_writer(config.name_vent, config.easting, config.northing)

In [ ]:
csv_data = file_opener.csv_vent_reader()

In [ ]:
# [grid_parameters]
if config.grid_mode == 'yes':
    config.ventgrid_size = 375
    config.ventgrid_resolution = 375
    config.dem_resolution = 5
    config.path_to_grid_folder = config.set_path_to_grid_folder() # *Need path_to_eruptions and name_vent
    config.show(config.grid_mode_summary()) 
else:
    print("No grid mode.")

In [ ]:
if config.grid_mode == 'yes':
    # Make a new csv file with the location of each new vent
    grid = grid_maker_reader(csv_data, config)
else:
    if not config.from_vent:
        # Read the new csv
        csv_data = file_opener.csv_vent_reader()

In [ ]:
# Read the new csv
csv_data = file_opener.csv_vent_reader()
print(DataFrame(csv_data))

In [ ]:
runner = Runner(config)

In [ ]:
# Start the loop to run downflowgo for each row in the csv file
for data in csv_data:
    main_id = data["flow_id"]
    runner.run_model(data, main_id)
    clear_output(wait=True)

In [ ]:
if config.grid_mode == 'yes':
    runner.run_gridmode(grid)
else:
    print("No grid mode.")

In [ ]:
# [mapping]
config.mapping_display = "yes"

config.img_tif_map_background_path = "./data/inputs/map_background/HS_srtm_48_17_utm.tif"
config.monitoring_network_path =  "./data/inputs/monitoring_stations/example_monitoring_stations.shp"
config.lava_flow_outline_path = "./data/inputs/lavaflow_outline/example_lavaflow_outline.shp"
config.logo_path = "./data/inputs/logo/all_logo.png"
config.source_img_tif_map_background = "©credit"
config.unverified_data = "NON VALIDES NE PAS DIFFUSER"
config.set_map_layers()

In [ ]:
# Map parameter
print("Display: " + config.mapping_display)
config.show(config.map_layers)

In [ ]:
# Make the map
mapping = Mapping(path_to_folder = runner.path_to_folder, 
                    dem = config.dem,
                    flow_id = config.name_vent,
                    map_layers = config.map_layers,
                    sim_layers = runner.sim_layers,
                    mode = config.mode,
                    language = config.language,
                    grid_mode = config.grid_mode)
mapping.create_map(display = config.mapping_display)

In [ ]:
print("************************************** THE END *************************************")

runtime(start_time, time.time())